# AI Model Experiment & Evaluation
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membandingkan dua pendekatan AI dalam menyelesaikan task sentiment analysis ulasan pelanggan:
1. Model klasik Machine Learning (Scikit-learn)
2. LLM API (Gemini)

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

> 🔧 **Catatan Penyesuaian Dataset**
>
> Notebook ini menggunakan dataset `customer_reviews_sentiment.csv` dengan kolom `review_text` dan `sentiment` (nilai: `positif`/`negatif`).
>
> Apabila dataset final berbeda dari yang digunakan saat ini, sesuaikan bagian berikut:
> - Nama file pada `pd.read_csv(...)` di Section 2
> - Nama kolom teks dan label pada Section 3 (saat ini: `review_text`, `sentiment`)
> - Label yang diminta pada prompt LLM di Section 5.2 (saat ini: `positif`/`negatif`)
> - Studi Kasus pada Assignment Brief, apabila domain data berbeda dari e-commerce

## 1. Problem Statement

_Tuliskan pemahamanmu terhadap use case di sini (objective, target/label, batasan/asumsi)._

## 2. Import Library & Load Dataset

In [27]:
# 🔧 Sesuaikan nama file jika dataset final berbeda
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import ollama
import os

df = pd.read_csv('../data/customer_reviews_sentiment.csv')
df.head()

,review_id,product_name,review_text,sentiment
0,1,Kemeja Flanel,Pelayanan lambat dan tidak responsif saat diko...,negatif
1,2,Case HP,"Puas banget belanja disini, proses cepat dan b...",positif
2,3,Rice Cooker,"Terima kasih seller, barangnya awet dan sesuai...",positif
3,4,Case HP,Warna produk berbeda jauh dari foto di listing.,negatif
4,5,Headset Bluetooth,"Terima kasih seller, barangnya awet dan sesuai...",positif


## 3. Menyiapkan Train/Test Split

Pastikan test set yang sama digunakan untuk kedua pendekatan agar perbandingan adil.

In [28]:
# 🔧 Sesuaikan nama kolom jika dataset final menggunakan nama kolom berbeda
X = df['review_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))

Train size: 160
Test size: 40


## 4. Pendekatan 1 — Model Klasik (Scikit-learn)

### 4.1 Preprocessing & Feature Extraction

In [29]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

### 4.2 Training Model

In [30]:
clf = LogisticRegression()
clf.fit(X_train_vec, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

### 4.3 Prediksi pada Test Set

In [31]:
y_pred_classic = clf.predict(X_test_vec)

## 5. Pendekatan 2 — LLM OPEN SOURCE (Ollama, model = gpt-oss:20b)

### 5.1 Setup API

In [ ]:
# Tes Ollama LLM
print(ollama.chat(model='gpt-oss:20b',
                  messages=[{'role': 'user', 'content': 'Jawab satu kata: halo'}])['message']['content'])

Halo


### 5.2 Merancang Prompt

_Tuliskan prompt yang kamu rancang di sini, dan jelaskan alasannya (zero-shot / few-shot)._

In [34]:
# 🔧 Sesuaikan label pada prompt jika dataset final menggunakan label berbeda
import time

def classify_sentiment_llm(review_text, max_retries=3):
    prompt = f'''Klasifikasikan sentimen ulasan berikut sebagai "positif" atau "negatif".
Jawab hanya dengan satu kata: positif atau negatif.

Ulasan: "{review_text}"
Sentimen:
'''
    for attempt in range(max_retries):
        try:
            response = ollama.chat(
                model='gpt-oss:20b',
                messages=[{'role': 'user', 'content': prompt}]
            )
            return (response['message']['content'] or '').strip()
        except Exception as e:
            print(f'Error (percobaan {attempt+1}): {e}')
            time.sleep(2 ** (attempt + 1))   # tunggu 2s, 4s, 8s
    return ''

### 5.3 Menjalankan Prediksi pada Test Set

Catatan: lakukan normalisasi terhadap output LLM sebelum dibandingkan dengan label asli (misal: lowercase, strip whitespace).

In [35]:
import json, os, time

def normalize_label(raw):
    if not isinstance(raw, str):
        return 'tidak_valid'
    t = raw.strip().lower().strip('.!*"\' \n')
    if t == 'negatif':
        return 'negatif'
    if t == 'positif':
        return 'positif'
    return 'tidak_valid'

CACHE_PATH = '../data/llm_cache_ollama.json'
cache_raw = json.load(open(CACHE_PATH, encoding='utf-8')) if os.path.exists(CACHE_PATH) else {}

unique_texts = list(X_test.unique())
todo = [t for t in unique_texts if t not in cache_raw]
print(f'Teks unik: {len(unique_texts)} | perlu dipanggil: {len(todo)}')

start = time.time()
for i, text in enumerate(todo):
    cache_raw[text] = classify_sentiment_llm(text)
    with open(CACHE_PATH, 'w', encoding='utf-8') as f:
        json.dump(cache_raw, f, ensure_ascii=False, indent=2)
    print(f'{i+1}/{len(todo)} ({time.time()-start:.0f} dtk): {cache_raw[text]!r}')

raw_outputs = [cache_raw[t] for t in X_test]
y_pred_llm = [normalize_label(r) for r in raw_outputs]

pd.DataFrame({'review': X_test.values, 'true': y_test.values,
              'raw_llm': raw_outputs, 'pred_llm': y_pred_llm}
            ).to_csv('../data/llm_predictions.csv', index=False)

print('Total review       :', len(X_test))
print('Valid predictions  :', len(y_pred_llm) - y_pred_llm.count('tidak_valid'))
print('Invalid predictions:', y_pred_llm.count('tidak_valid'))

Teks unik: 26 | perlu dipanggil: 26
1/26 (11 dtk): 'negatif'
2/26 (16 dtk): 'positif'
3/26 (19 dtk): 'positif'
4/26 (22 dtk): 'positif'
5/26 (25 dtk): 'negatif'
6/26 (28 dtk): 'negatif'
7/26 (31 dtk): 'positif'
8/26 (34 dtk): 'negatif'
9/26 (37 dtk): 'negatif'
10/26 (39 dtk): 'negatif'
11/26 (43 dtk): 'positif'
12/26 (46 dtk): 'positif'
13/26 (50 dtk): 'negatif'
14/26 (53 dtk): 'positif'
15/26 (56 dtk): 'negatif'
16/26 (62 dtk): 'negatif'
17/26 (64 dtk): 'positif'
18/26 (67 dtk): 'negatif'
19/26 (70 dtk): 'positif'
20/26 (73 dtk): 'positif'
21/26 (75 dtk): 'positif'
22/26 (80 dtk): 'negatif'
23/26 (84 dtk): 'positif'
24/26 (86 dtk): 'negatif'
25/26 (88 dtk): 'negatif'
26/26 (91 dtk): 'positif'
Total review       : 40
Valid predictions  : 40
Invalid predictions: 0


## 6. Evaluasi dan Perbandingan

### 6.1 Evaluasi Model Klasik

In [36]:
print('=== Model Klasik (Scikit-learn) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_classic))
print(classification_report(y_test, y_pred_classic))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_classic))

=== Model Klasik (Scikit-learn) ===
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00        18
     positif       1.00      1.00      1.00        22

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix:
 [[18  0]
 [ 0 22]]


### 6.2 Evaluasi LLM API

In [38]:
print('=== LLM API (Ollama: gpt-oss:20b) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_llm))
print(classification_report(y_test, y_pred_llm))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_llm))

=== LLM API (Ollama: gpt-oss:20b) ===
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00        18
     positif       1.00      1.00      1.00        22

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix:
 [[18  0]
 [ 0 22]]


### 6.3 Tabel Perbandingan Ringkasan

_Susun tabel ringkasan (bisa markdown table atau DataFrame) yang membandingkan Accuracy, Precision, Recall, F1-Score kedua pendekatan._

In [40]:
import time

# LLM
t0 = time.time()
_ = [classify_sentiment_llm(t) for t in X_test.unique()]
t_llm = (time.time() - t0) / len(X_test.unique())

# Klasik ML - TF-IDF + Logistic Regression
t0 = time.time()
_ = clf.predict(vectorizer.transform(X_test.unique()))
t_clf = (time.time() - t0) / len(X_test.unique())

print(f'Klasik: {t_clf*1000:.2f} ms/ulasan | LLM: {t_llm*1000:.0f} ms/ulasan')

Klasik: 0.12 ms/ulasan | LLM: 3368 ms/ulasan


In [41]:
hard = pd.DataFrame({
    'review': [
        'Barangnya tidak jelek, lumayan lah.',
        'Bagus banget, baru sehari sudah patah. Mantap sekali.',
        'Mantul gan, auto langganan!',
        'Zonk parah, nyesel beli.',
        'Kurir ramah, tapi barangnya cacat dan saya kecewa.',
        'Tidak mengecewakan sama sekali, malah melebihi harapan.',
        'Cakep dipakai, awet juga.',
        'Lama banget nyampenya, packing juga asal.',
    ],
    'true': ['positif','negatif','positif','negatif','negatif','positif','positif','negatif']
})

hard['klasik'] = clf.predict(vectorizer.transform(hard['review']))
hard['llm'] = [normalize_label(classify_sentiment_llm(t)) for t in hard['review']]

display(hard)
print('Akurasi klasik:', (hard['klasik'] == hard['true']).mean())
print('Akurasi LLM   :', (hard['llm'] == hard['true']).mean())

,review,true,klasik,llm
0,"Barangnya tidak jelek, lumayan lah.",positif,negatif,positif
1,"Bagus banget, baru sehari sudah patah. Mantap ...",negatif,negatif,negatif
2,"Mantul gan, auto langganan!",positif,positif,positif
3,"Zonk parah, nyesel beli.",negatif,positif,negatif
4,"Kurir ramah, tapi barangnya cacat dan saya kec...",negatif,positif,negatif
5,"Tidak mengecewakan sama sekali, malah melebihi...",positif,negatif,positif
6,"Cakep dipakai, awet juga.",positif,positif,positif
7,"Lama banget nyampenya, packing juga asal.",negatif,positif,negatif


Akurasi klasik: 0.375
Akurasi LLM   : 1.0


## 7. Analisis Trade-off dan Limitation

_Tuliskan analisis: pendekatan mana yang lebih unggul dari sisi performa, trade-off effort/kecepatan/biaya, dan keterbatasan masing-masing pendekatan._

## 7. Analisis Trade-off dan Limitation

### 7.1 Performa

| Evaluasi | Klasik (TF-IDF + LogReg) | LLM (gpt-oss:20b via Ollama) |
|---|---|---|
| Test set dataset (40 ulasan) | Accuracy 1.00 | Accuracy 1.00 |
| Evaluasi dedup, 10 seed acak (test 25%) | rata-rata 0.70 (rentang 0.50 - 0.90) | tidak diuji |
| Kasus sulit tambahan (8 kalimat) | Accuracy 0.375 | Accuracy 1.00 |

Pada test set dataset, kedua pendekatan seri di 100%, sehingga tidak ada pemenang dari
angka ini saja. Skor model klasik tidak bisa dipercaya sepenuhnya karena dataset hanya berisi
40 kalimat unik yang diulang hingga 200 baris, dan 39 dari 40 teks di test set juga muncul
di train set (data leakage). Model klasik pada dasarnya menghafal kalimat, bukan memahami
sentimen. Ketika duplikat dibuang dan diuji pada kalimat yang belum pernah dilihat, akurasinya
turun menjadi sekitar 70% dan tidak stabil antar split.

Pada 8 kalimat tambahan yang sengaja dibuat sulit (negasi, sarkasme, slang, kampanye
kosakata baru), model klasik hanya benar 3 dari 8 (37,5%, di bawah tebakan acak),
sedangkan LLM benar 8 dari 8. Kesimpulannya, **LLM jauh lebih mampu menggeneralisasi ke
kalimat baru**. Catatan: sampelnya hanya 8 kalimat dengan label buatan sendiri, jadi hasil ini
bersifat indikatif, bukan konklusif.

Interpretasi metrik pada test set dataset:
- Precision 1.00 pada kedua kelas: setiap ulasan yang diprediksi positif/negatif memang benar.
- Recall 1.00 pada kedua kelas: tidak ada ulasan yang terlewat. Recall kelas negatif penting
  karena ulasan negatif yang lolos membuat tim produk terlambat mengetahui masalah.
- Confusion matrix tanpa kesalahan (FP = 0, FN = 0). Karena ini hasil pada data yang mudah dan
  bocor, angka sempurna tidak boleh dibaca sebagai "model sempurna".

### 7.2 Trade-off

| Aspek | Klasik | LLM lokal (gpt-oss:20b) |
|---|---|---|
| Effort implementasi | Perlu menyiapkan data berlabel dan training | Tanpa training, cukup merancang prompt |
| Kecepatan inferensi | 0,12 ms per ulasan | sekitar 3.368 ms per ulasan (sekitar 28.000x lebih lambat) |
| Biaya | Hampir nol saat inferensi | Tanpa biaya per request, tetapi butuh hardware kuat (sekitar 16 GB RAM/VRAM). Jika memakai API berbayar, biaya naik seiring volume |
| Skalabilitas | 1 juta ulasan sekitar 2 menit | 1 juta ulasan sekitar 39 hari bila berurutan di satu mesin (perkiraan kasar dari angka di atas) |
| Konsistensi output | Selalu valid | Teks bebas, perlu normalisasi, ada risiko `tidak_valid` ([isi jumlah] dari [isi total] pada eksperimen ini) |
| Ketergantungan | Mandiri | Bergantung pada model dan infrastruktur yang dipakai |

Pengalaman selama eksperimen juga memberi bukti nyata untuk trade-off LLM API: free tier
Gemini yang sempat dicoba dibatasi 20 request per hari (error 429), sehingga tidak layak untuk
pemakaian nyata dan eksperimen dialihkan ke model lokal lewat Ollama. Ini menunjukkan risiko
batas kuota dan ketergantungan pada pihak ketiga. (Catatan: perbandingan dengan Gemini
sendiri tidak diselesaikan, perbandingan final memakai gpt-oss:20b.)

### 7.3 Limitation

**Model klasik**
- Rentan menghafal pada data repetitif, sehingga skor tinggi belum tentu mencerminkan kemampuan.
- Tidak menggeneralisasi ke kosakata baru, slang, atau pola yang tidak ada di data latih.
- Tidak memahami konteks seperti negasi dan sarkasme karena hanya melihat bobot kata.
- Kualitasnya sangat bergantung pada jumlah dan keragaman data. Hasil buruk di sini lebih
  disebabkan oleh data latih yang hanya 40 kalimat unik, bukan semata keterbatasan algoritma.

**LLM**
- Lambat dan butuh resource besar dibanding model klasik.
- Output berupa teks bebas sehingga perlu normalisasi dan penanganan `tidak_valid`.
- Hasil bisa berbeda antar versi model, dan prompt perlu dijaga konsisten.
- Evaluasi hanya pada ulasan yang pendek dan tegas, sehingga kemampuan pada ulasan campuran
  atau ambigu belum teruji.

**Eksperimen**
- Dataset sintetis: hanya 40 kalimat unik, tidak mewakili keragaman ulasan nyata.
- Test set kecil, satu kesalahan mengubah akurasi beberapa poin persentase.
- Kasus sulit tambahan hanya 8 kalimat dengan label buatan sendiri, dan satu atau dua di
  antaranya bisa diperdebatkan (misalnya ulasan bernada campuran).
- Pengukuran waktu klasik dilakukan batch sekaligus, sedangkan LLM per request, dan keduanya
  dijalankan di satu mesin lokal sehingga angkanya bergantung pada spesifikasi mesin tersebut.

## 8. Rekomendasi Technical Approach

**Rekomendasi: pendekatan hybrid, dengan model klasik sebagai lapisan utama dan LLM sebagai
lapisan kedua untuk ulasan yang tidak yakin.**

**Alasan berdasarkan hasil eksperimen:**
1. Model klasik sangat cepat (0,12 ms) dan murah, cocok menangani sebagian besar ulasan pada
   volume besar. LLM yang 28.000x lebih lambat tidak praktis untuk memproses seluruh ulasan.
2. Namun model klasik rapuh pada kalimat baru (37,5% pada kasus sulit, sekitar 70% pada
   evaluasi dedup), sehingga tidak aman dipakai sendirian di produksi.
3. LLM menunjukkan generalisasi yang jauh lebih baik (100% pada kasus sulit), jadi cocok
   sebagai "lapis kedua" untuk ulasan yang sulit.

**Rancangan singkat:**
- Ulasan masuk ke model klasik. Jika confidence (`predict_proba`) di atas ambang tertentu,
  hasilnya langsung dipakai.
- Jika confidence di bawah ambang, ulasan diteruskan ke LLM.
- Hasil LLM bisa disimpan sebagai data berlabel baru untuk melatih ulang model klasik
  secara berkala, sehingga porsi yang dikirim ke LLM makin kecil seiring waktu.

**Hal yang harus divalidasi sebelum diterapkan** (belum dibuktikan oleh eksperimen ini):
- Apakah kesalahan model klasik memang terkonsentrasi di ulasan dengan confidence rendah.
  Jika model klasik salah dengan confidence tinggi, hybrid ini tidak akan menangkapnya.
  [isi hasil cek `predict_proba` pada kasus sulit di sini]
- Penentuan ambang confidence dan proporsi ulasan yang akan diteruskan ke LLM, karena itu
  menentukan biaya dan latensi total.
- Evaluasi pada data ulasan nyata yang lebih besar dan beragam, bukan 40 kalimat unik.

**Kapan pilihan lain lebih tepat:**
- Jika volume kecil dan akurasi lebih penting daripada kecepatan, LLM saja sudah cukup.
- Jika tim bisa mengumpulkan ribuan ulasan berlabel yang beragam, model klasik yang dilatih
  ulang kemungkinan besar sudah cukup baik sehingga LLM hanya perlu untuk kasus khusus.

**Langkah lanjutan:** kumpulkan data ulasan nyata, bersihkan duplikat sebelum split, tentukan
ambang confidence, lalu ukur akurasi, latensi, dan biaya gabungan dari sistem hybrid.